# 16.1 最慢或最佔空間的是哪裡？

# 第 16 章：由量測帶出效率 BKM

前置：已有第4章能forward與生成的模型。像整理廚房，先找哪一步最慢再改工作流程。CPU小例只驗證數值與資料契約；GPU速度、峯值記憶體與編譯收益必須在讀者的硬體實測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：舊筆記不用重寫：只加新字的K/V**

cache保存每層筆記；不是提前知道下一字答案。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/cache.svg")))

**先想一想：**最大的參數層一定是最慢的層嗎？

只看總時間像知道整餐花多久，卻不知道是在切菜還是等烤箱。profiler把各運算花費列出；先看瓶頸，別一次打開所有優化。

**把直覺寫成數學：**時間、參數bytes、activation峯值是不同量；profiler本身也有開銷。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig
from torch.profiler import profile, ProfilerActivity

model = TinyLM(ModelConfig(width=8))
x = torch.tensor([[1, 2, 3]])
with profile(activities=[ProfilerActivity.CPU]) as p:
    model(x)
print(p.key_averages().table(sort_by="self_cpu_time_total", row_limit=5))

**如何讀結果：**只有CPU活動；GPU需CUDA activity與同步。先warmup，再測多次，避免將初始化當穩態。

**只改一件事：**只增加T，找attention相關成本是否增加。
